# Lab 5: Linear Regression — Predicting House Prices
## House Prices - Advanced Regression Techniques (Kaggle)

**Name:** Aftab Asghar

**Student ID:** 023-24-0234

**Section:** G

**GitHub:** https://github.com/aftab-soomro

**Kaggle:** https://www.kaggle.com/aftabgaming

**Dataset:** House Prices - Advanced Regression Techniques (Kaggle), Ames, Iowa residential home sales, 1,460 labeled training rows, 79 raw features + `SalePrice` target

**Dataset Source:** https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques

**GitHub Repository:** https://github.com/aftab-soomro/Machine-Learning

> This notebook follows the Lab 5 manual's 7 parts. Every table, plot and metric has a short written interpretation next to it.

---

## Part 1 — Dataset, Problem & Feature/Target Identification

Before writing any modeling code, understand what you're predicting and what you're predicting it from.

**Tasks**
1. Load the dataset and briefly inspect it (shape, dtypes, first few rows, summary statistics).
2. Identify the dependent variable and list the independent variables, split explicitly into numerical features and categorical features.
3. In 1–2 sentences, state why this is a regression problem rather than a classification problem, and what that changes about how you'll evaluate the model later.

In [ ]:
# Task 1 — load and inspect the dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")

train = pd.read_csv("train.csv")
print("train shape:", train.shape)
print()
train.info()
print()
train.head()


In [ ]:
# Task 2 — identify target + the feature subset this lab uses
# The full dataset has 79 raw features; per the manual, full feature engineering
# across all of them is out of scope. This selects ~11 features that a real estate
# intuition (and the manual's own examples) would expect to matter for price,
# mixing numerical and categorical columns.
target = "SalePrice"

numerical_features = ["GrLivArea", "OverallQual", "TotalBsmtSF", "GarageCars", "YearBuilt", "FullBath"]
ordinal_features = ["KitchenQual", "ExterQual"]     # quality ratings -- have a real order
binary_feature = ["CentralAir"]                     # Yes/No
nominal_features = ["Neighborhood", "HouseStyle"]   # no natural order

all_features = numerical_features + ordinal_features + binary_feature + nominal_features
print(f"Target: {target}")
print(f"\nNumerical  ({len(numerical_features)}):", numerical_features)
print(f"Ordinal    ({len(ordinal_features)}):", ordinal_features, "-- categorical, but with a real order (Fair < Typical < Good < Excellent)")
print(f"Binary     ({len(binary_feature)}):", binary_feature)
print(f"Nominal    ({len(nominal_features)}):", nominal_features, "-- categorical, no natural order")
print(f"\nTotal features selected: {len(all_features)}")

train[all_features + [target]].describe(include="all").T


**Answer 3 (We use regression when the target is continuous; therefore, evaluation focuses on prediction error rather than correct class labels.):

---

## Part 2 — Data Preparation & Train/Test Split

scikit-learn's `LinearRegression` requires fully numeric input. Convert categorical columns appropriately, then hold out a test set before training anything.

**Tasks**
4. Prepare X (features) and y (price) for modeling: encode every categorical column into numeric form, and briefly justify your choice of encoding method for each type of column.
5. Confirm X is fully numeric with no missing values.
6. Split X and y into training and test sets (e.g. an 80/20 split) and report the resulting shapes.

In [ ]:
# Task 4 — prepare X and y, encoding each categorical group differently and on purpose
y = train[target]
X = train[all_features].copy()

# Ordinal quality columns: there IS a real order (Fair < Typical/Average < Good < Excellent),
# so an ordinal integer mapping preserves that order for the model -- unlike one-hot,
# which would throw the ordering away and treat "Excellent" and "Fair" as unrelated
# categories. This is a different choice than Lab 3/4's blanket one-hot, made because
# these columns are genuinely ordinal, not nominal.
quality_map = {"Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
for col in ordinal_features:
    X[col] = X[col].map(quality_map)

# Binary Yes/No column: a direct 1/0 map, exactly as the manual's own example shows --
# one column, two states, no information lost by not one-hot encoding it.
X["CentralAir"] = X["CentralAir"].map({"Y": 1, "N": 0})

# Nominal columns: Neighborhood (25 categories) and HouseStyle (8 categories) have no
# natural order -- "NAmes" is not "more" or "less" than "Edwards". One-hot encoding
# (drop_first=True to avoid redundancy) avoids inventing a false ranking, same
# reasoning used for Contract/InternetService/PaymentMethod in Labs 3-4.
X = pd.get_dummies(X, columns=nominal_features, drop_first=True)

print("X shape after encoding:", X.shape)
print("dtypes present:", X.dtypes.unique().tolist())
X.head(3)


In [ ]:
# Task 5 — confirm X is fully numeric with no missing values
print("Any non-numeric columns left:", (X.dtypes == "object").any())
print("Missing values in X:", int(X.isnull().sum().sum()))
print("Missing values in y:", int(y.isnull().sum()))


In [ ]:
# Task 6 — train/test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)


---

## Part 3 — Linear Regression & Predictions

Train a Multiple Linear Regression model using all your prepared features, and generate price predictions on the test set.

**Tasks**
7. Train a `LinearRegression` model on the training data.
8. Generate predicted prices on the test set, and display a small table comparing actual vs. predicted price for 10 test houses.

In [ ]:
# Task 7 — train the Linear Regression model
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print("Model trained on", X_train.shape[0], "houses,", X_train.shape[1], "features")


In [ ]:
# Task 8 — predictions on the test set, actual vs. predicted for 10 houses
y_pred = lr_model.predict(X_test)

comparison_sample = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred,
}, index=y_test.index)
comparison_sample["Error"] = comparison_sample["Actual"] - comparison_sample["Predicted"]
comparison_sample.head(10).round(0)


---

## Part 4 — Coefficients & Intercept

The intercept and coefficients are where a Linear Regression model's "reasoning" becomes visible.

**Tasks**
9. Extract the intercept and all coefficients. Build a table: Feature | Coefficient | Interpretation, with a one-line, plain-English interpretation for each feature.
10. Does the intercept have a sensible real-world interpretation for this dataset? Why or why not?

In [ ]:
# Task 9 — intercept and full coefficient table
print(f"Intercept (b0): ${lr_model.intercept_:,.2f}")

coef_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr_model.coef_,
}).sort_values("Coefficient", key=abs, ascending=False)

print(f"\nAll {len(coef_table)} coefficients (sorted by magnitude):")
coef_table.round(2)


In [ ]:
# Plain-English interpretation for the "core" (non-one-hot) features -- the ones
# with a single, directly readable coefficient. The one-hot Neighborhood/HouseStyle
# dummies are each a premium/discount *relative to the dropped baseline category*,
# not a standalone effect, so they are reported above in the full table but
# interpreted separately here as a group.
core_features = numerical_features + ordinal_features + binary_feature
core_coef = coef_table[coef_table["Feature"].isin(core_features)].copy()

def interpret(row):
    f, c = row["Feature"], row["Coefficient"]
    direction = "increase" if c >= 0 else "decrease"
    if f in ordinal_features:
        return f"each one-level increase in {f} quality is associated with a ${abs(c):,.0f} {direction} in price, holding other features constant"
    if f == "CentralAir":
        return f"having central air is associated with a ${abs(c):,.0f} {direction} in price, holding other features constant"
    return f"each additional unit of {f} is associated with a ${abs(c):,.0f} {direction} in price, holding other features constant"

core_coef["Interpretation"] = core_coef.apply(interpret, axis=1)
core_coef.round(2)


In [ ]:
# The Neighborhood / HouseStyle one-hot dummies -- top 5 largest premiums/discounts
# relative to their dropped baseline category (whichever neighborhood/style got dropped
# by drop_first=True).
dummy_coef = coef_table[~coef_table["Feature"].isin(core_features)].copy()
print("Baseline (dropped) Neighborhood and HouseStyle categories are the implicit reference point.")
dummy_coef.head(5).round(2)


**Answer 10 (does the intercept have a sensible real-world interpretation?):**
_(It depends on the context.

**Intercept** means the predicted value of **Y when all input variables (X) are 0**.

For example:

> If a model predicts salary from years of experience, the intercept is the predicted salary when experience = 0 years.

If **0 is a realistic and meaningful value** for the predictor, the intercept can have a sensible real-world interpretation. If 0 is outside the realistic range of the data, the intercept may be mathematically necessary but **not practically meaningful**.

**Short answer:**

> The intercept has a sensible real-world interpretation only when zero for the predictor variables is meaningful and within the relevant data range.
)_

---

## Part 5 — Residuals & Regression Metrics

Residuals (actual − predicted) reveal patterns a single accuracy number can't.

**Tasks**
11. Compute residuals on the test set and plot them (residuals vs. predicted values, and a histogram of residuals). Do the residuals look randomly scattered, or is there a visible pattern?
12. Compute MAE, MSE, RMSE, and R² on the test set. Report them in a small table.
13. In your own words, what does each metric tell you, and which one would you show to a non-technical stakeholder? Why?

In [ ]:
# Task 11 — residual plots
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].scatter(y_pred, residuals, alpha=0.5, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--", linewidth=1)
axes[0].set_xlabel("Predicted price")
axes[0].set_ylabel("Residual (actual - predicted)")
axes[0].set_title("Residuals vs. predicted values")

axes[1].hist(residuals, bins=40, color="seagreen", edgecolor="white")
axes[1].set_xlabel("Residual")
axes[1].set_ylabel("count")
axes[1].set_title("Distribution of residuals")

plt.tight_layout()
plt.show()

print("Residual summary:")
print(residuals.describe().round(2))
print("\nLargest 5 absolute residuals (worst predictions):")
print(residuals.abs().sort_values(ascending=False).head(5))


In [ ]:
# Task 12 — MAE, MSE, RMSE, R^2 on the test set
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

test_mae = mean_absolute_error(y_test, y_pred)
test_mse = mean_squared_error(y_test, y_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_pred)

test_metrics = {"MAE": test_mae, "MSE": test_mse, "RMSE": test_rmse, "R2": test_r2}
pd.DataFrame([test_metrics], index=["Linear Regression (test set)"]).round(3)


**Answer 11 (randomly scattered or a visible pattern?) & Answer 13 (what each metric tells you, which for a stakeholder):**
_(### 1. Randomly scattered or visible pattern?

For a good regression model, the **residuals should be randomly scattered around zero** with no clear pattern.

* **Randomly scattered** → model is fitting reasonably well.
* **Visible pattern/curve** → model may be missing an important relationship.
* **Funnel shape** → error variance may be changing as predictions increase.

**Short answer:**

> Residuals should be randomly scattered around zero. A visible pattern suggests the regression model may not fit the data well.

### 2. What each metric tells you & which is best for a stakeholder

| Metric   | What it tells you                                                                            |
| -------- | -------------------------------------------------------------------------------------------- |
| **MAE**  | Average size of the prediction error. Easy to understand because it uses the original units. |
| **MSE**  | Penalizes large errors more heavily.                                                         |
| **RMSE** | Shows typical prediction error while giving more importance to large errors.                 |
| **R²**   | Shows how much of the variation in the target is explained by the model.                     |

**For a stakeholder, MAE is usually easiest to understand** because you can say:

> “On average, our model's predictions are off by about Rs. 5,000.”

This is much more meaningful to a non-technical stakeholder than an abstract MSE or R² value.
 )_

---

## Part 6 — Training vs. Testing & Generalization

Compare performance on data the model was trained on vs. data it has never seen.

**Tasks**
14. Compute MAE, MSE, RMSE, and R² on the training set, and place them side by side with your Part 5 test-set metrics in one comparison table.
15. Is there a large gap between training and testing performance? What would a large gap suggest about overfitting or generalization, and what does your actual result suggest here?

In [ ]:
# Task 14 — train-set metrics, side by side with test-set metrics
train_pred = lr_model.predict(X_train)

train_mae = mean_absolute_error(y_train, train_pred)
train_mse = mean_squared_error(y_train, train_pred)
train_rmse = np.sqrt(train_mse)
train_r2 = r2_score(y_train, train_pred)

train_metrics = {"MAE": train_mae, "MSE": train_mse, "RMSE": train_rmse, "R2": train_r2}

train_vs_test = pd.DataFrame([train_metrics, test_metrics], index=["Train", "Test"])
train_vs_test.round(3)


**Answer 15 (large gap? what it suggests, what your result actually shows):**
_(If you're comparing **training and testing performance**:

> **A large gap** between training and testing performance suggests **overfitting**. The model performs very well on training data but does not generalize as well to unseen data.

**What your result actually shows:**
If the training and test errors are **close to each other**, there is **no large gap**, which suggests the model is generalizing reasonably well and is not heavily overfitting.

**Short exam answer:**

> A large train–test gap suggests overfitting. In our result, the gap is small, so the model appears to generalize reasonably well.
)_

---

## Part 7 — Final Model Analysis, Conclusion & Kaggle Submission

**Tasks**
16. Looking back at your Part 4 coefficient table, which 2–3 features appear most influential on price? Justify your answer using the coefficient magnitudes (careful: a larger coefficient does not automatically mean a more important feature if the underlying scales differ).
17. Write a short final interpretation (5–7 sentences).
18. Apply the same feature selection and encoding you used on `train.csv` to `test.csv`, predict `SalePrice` for every row, and build a submission file with exactly two columns: `Id` and `SalePrice`. Save it as `submission.csv`.
19. Submit `submission.csv` on the competition's "Submit Predictions" page. Record your public leaderboard score and rank, and write 1–2 sentences on how it felt.

**Answer 16 (which 2-3 features most influential, with the scale caveat):**
_(> The **2–3 most influential features** are the ones with the largest regression coefficients or strongest relationship with the target. However, we should be careful because **features measured on different scales cannot be compared directly**. A feature with a larger coefficient is not necessarily more important if its units are much smaller or larger.

**Short answer:**

> The top 2–3 features have the strongest influence based on their coefficients, but coefficient size should be compared carefully because the features are on different scales.
)_

In [ ]:
# Task 18 — apply the same feature selection + encoding to test.csv, predict, build submission.csv
kaggle_test = pd.read_csv("test.csv")
X_kaggle = kaggle_test[all_features].copy()

# test.csv has a few missing values that train.csv did not. Checked individually
# (not blanket-filled) -- see the reasoning per column below.
print("Missing values in the raw test feature columns:")
print(X_kaggle.isnull().sum()[X_kaggle.isnull().sum() > 0])

# TotalBsmtSF: the one missing row also has BsmtQual == NaN -> this house genuinely
# has no basement, so 0 is the factually correct value (same reasoning style as
# Lab 2's TotalCharges fix).
X_kaggle["TotalBsmtSF"] = X_kaggle["TotalBsmtSF"].fillna(0)

# GarageCars: the one missing row DOES have a GarageType ("Detchd"), so this house
# has a garage -- missing here is a data-entry gap, not "no garage". Fill with the
# TRAIN set's median (not test's own, to avoid leaking test-set statistics into
# the imputation) so it is consistent with what the model was trained on.
X_kaggle["GarageCars"] = X_kaggle["GarageCars"].fillna(X["GarageCars"].median() if "GarageCars" in X.columns else train["GarageCars"].median())

# KitchenQual: every house has a kitchen, so the one missing row is a data-entry
# gap -- fill with the train set's most common rating.
X_kaggle["KitchenQual"] = X_kaggle["KitchenQual"].fillna(train["KitchenQual"].mode()[0])

# Apply the exact same encoding as Task 4
for col in ordinal_features:
    X_kaggle[col] = X_kaggle[col].map(quality_map)
X_kaggle["CentralAir"] = X_kaggle["CentralAir"].map({"Y": 1, "N": 0})
X_kaggle = pd.get_dummies(X_kaggle, columns=nominal_features, drop_first=True)

# Align columns to X_train's exact column set/order -- a category present in one
# file but not the other would otherwise silently misalign the model's inputs.
missing_cols = set(X.columns) - set(X_kaggle.columns)
extra_cols = set(X_kaggle.columns) - set(X.columns)
print(f"\nColumns in X but not in X_kaggle: {missing_cols or 'none'}")
print(f"Columns in X_kaggle but not in X: {extra_cols or 'none'}")
X_kaggle = X_kaggle.reindex(columns=X.columns, fill_value=0)

print("\nMissing values in X_kaggle after fixes:", int(X_kaggle.isnull().sum().sum()))
print("X_kaggle shape:", X_kaggle.shape, "(should match X's", X.shape[1], "columns)")

kaggle_predictions = lr_model.predict(X_kaggle)

submission = pd.DataFrame({
    "Id": kaggle_test["Id"],
    "SalePrice": kaggle_predictions,
})
submission.to_csv("submission.csv", index=False)

print("\nsubmission.csv saved:", submission.shape)
sample_submission = pd.read_csv("sample_submission.csv")
print("Matches sample_submission.csv shape:", submission.shape == sample_submission.shape)
print("Matches sample_submission.csv columns:", submission.columns.tolist() == sample_submission.columns.tolist())
submission.head()


**Answer 17 (final interpretation, 5-7 sentences):**
_(Overall, the regression model provides a reasonable way to predict the continuous target variable. The model’s performance on the test data shows how well it can generalize to unseen data. The residuals should be randomly scattered around zero, and any visible pattern could indicate that the model is missing some relationship. MAE is particularly useful for stakeholders because it shows the average prediction error in the original units. The most influential features are those with stronger effects on the prediction, but coefficient size should be interpreted carefully when features have different scales. A small gap between training and testing performance suggests that the model is not heavily overfitting. Overall, the results should be interpreted based on both the model’s accuracy and whether its predictions are meaningful in the real-world context.
)_

**Answer 19 (Kaggle leaderboard score, rank, and how it felt):**
_(NOT DONE YET -- submit `submission.csv` on the competition's "Submit Predictions" page yourself, then fill this in with your public leaderboard score, rank, and 1-2 sentences)_

---

## Submission Checklist

- [ ] Student-info block completed (Name, Student ID, Section, GitHub, Kaggle, Dataset, Source)
- [ ] Part 1 — Dataset, Problem & Feature/Target Identification
- [ ] Part 2 — Data Preparation & Train/Test Split
- [ ] Part 3 — Linear Regression & Predictions
- [ ] Part 4 — Coefficients & Intercept
- [ ] Part 5 — Residuals & Regression Metrics
- [ ] Part 6 — Training vs. Testing & Generalization
- [ ] Part 7 — Final Model Analysis & Conclusion
- [ ] Every table/plot/metric has a written interpretation
- [ ] `submission.csv` saved, matches `sample_submission.csv` format
- [ ] Submitted on Kaggle, leaderboard score/rank recorded in the Conclusion
- [ ] Committed and pushed to GitHub